# LangChain client

Integrating Llama Stack with LangChain is a great way to build agentic workflows or RAG (Retrieval-Augmented Generation) applications. Since Llama Stack provides an OpenAI-compatible interface, we can use the ChatOpenAI class or the dedicated langchain-community integration.

Solution Overview
In this version, we will:

Install the LangChain community and OpenAI integration packages.

Configure the ChatOpenAI client to point to the Llama Stack local endpoint.

Invoke a chain using a template to show how LangChain manages the prompt and response cycle.

Implementation Guide
1. Environment Setup
You will need the LangChain core and OpenAI integration libraries:

In [1]:
%pip install -U langchain-openai langchain-core

Note: you may need to restart the kernel to use updated packages.


2. List the models

In [12]:
from openai import OpenAI

client = OpenAI(base_url=URL, api_key="not-needed")

models = client.models.list()
for model in models:
    print(model.id)

vllm-inference-1/redhataillama-31-8b-instruct
vllm-inference/redhataillama-31-8b-instruct
vllm-inference-2/granite-3-2-8b-instruct


3. The Jupyter Notebook Code
The trick here is setting the base_url to your Llama Stack endpoint (usually port 8321) and appending /v1 to match the OpenAI-style routing.

In [17]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Note the '/v1' suffix for OpenAI compatibility
# URL = "https://granite-31-8b-instruct-rh-ai-poc.apps.ocp-svil.bluarancio.com/v1"
# URL = "http://redhataillama-31-8b-instruct-predictor.rh-ai-poc.svc.cluster.local:8080/v1"

URL = "http://llamastack-custom-distribution-service.rh-ai-poc.svc.cluster.local:8321/v1"

# 1. Setup the Llama Stack LLM
# Llama Stack is OpenAI-compatible. 
# 'api_key' is required by the class but ignored by local Llama Stack.

llm = ChatOpenAI(
    base_url=URL,
    api_key="llama-stack-dummy-key",
    model="vllm-inference-2/granite-3-2-8b-instruct", # Match your local model ID
    temperature=0.1
)

# 2. Define a Prompt Template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an helpful assistant."),
    ("user", "{input}")
])

# 3. Create a Simple Chain
# We pipe the prompt to the model, then parse the result to a string.
chain = prompt | llm | StrOutputParser()

# 4. Run the Query
try:
    question = "Hello"
    print(f"Querying Llama Stack via LangChain...\n")
    
    response = chain.invoke({"input": question})
    
    print("--- Response ---")
    print(response)

except Exception as e:
    print(f"Error connecting to Llama Stack: {e}")
    print("Ensure your Llama Stack server is running at the base_url provided.")

Querying Llama Stack via LangChain...

--- Response ---
Hello! How can I assist you today?
